In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════════╗
# ║  SECTION 0: API DATA INGESTION (with Sprint Race Support)                  ║
# ║  Paste this cell at the TOP of your notebook, above Section 1 (Setup)      ║
# ║                                                                              ║
# ║  Each race weekend:                                                          ║
# ║    1. Update ROUND, TRACK_ID, and IS_SPRINT_WEEKEND below                  ║
# ║    2. Run this cell                                                          ║
# ║    3. Run the rest of the notebook                                           ║
# ╚══════════════════════════════════════════════════════════════════════════════╝

import requests
import pandas as pd
import os
import time

# ── CONFIGURATION — UPDATE THESE EACH RACE WEEKEND ───────────────────────────
SEASON            = 2026
ROUND             = 3            # Change to current round number
TRACK_ID          = 'japan'      # Match your schema track_id slug
IS_SPRINT_WEEKEND = False         # Set to True for sprint weekends, False otherwise

# 2026 Sprint weekends for reference:
# Round 2  - China
# Round 4  - Bahrain
# Round 11 - Austria
# Round 18 - United States
# Round 21 - São Paulo
# Round 23 - Qatar

# ── PATH CONFIGURATION ────────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
RAW_DATA_PATH = '/content/drive/MyDrive/f1-2026-season/data/raw/'

# ── API BASE URL ──────────────────────────────────────────────────────────────
BASE_URL = 'http://api.jolpi.ca/ergast/f1'


# ── HELPER: CONVERT LAP TIME STRING TO SECONDS ───────────────────────────────
def lap_time_to_seconds(time_str):
    if not time_str or pd.isna(time_str):
        return None
    try:
        parts = time_str.strip().split(':')
        if len(parts) == 2:
            return round(int(parts[0]) * 60 + float(parts[1]), 3)
        elif len(parts) == 3:
            return round(int(parts[0]) * 3600 + int(parts[1]) * 60 + float(parts[2]), 3)
    except (ValueError, IndexError):
        return None


# ── HELPER: FETCH FROM API ────────────────────────────────────────────────────
def fetch_api(url):
    try:
        print(f"  Fetching: {url}")
        response = requests.get(url, timeout=10)
        response.raise_for_status()
        return response.json()
    except requests.exceptions.RequestException as e:
        print(f"  ✗ API call failed: {e}")
        return None


# ── HELPER: PARSE STATUS ──────────────────────────────────────────────────────
def parse_status(api_status, position_text):
    if api_status == 'Finished' or '+' in api_status or 'Lap' in api_status:
        return 'Finished', ''
    else:
        return 'DNF', api_status


# ── HELPER: BUILD RESULTS DATAFRAME ──────────────────────────────────────────
def build_results_df(results_list, race_info, session='Race'):
    """
    Parse API result list into a DataFrame matching our schema.
    Works for both Race and Sprint results — same API structure.
    The session column ('Race' or 'Sprint') lets us distinguish rows
    when they are stacked together in the season master DataFrame.
    """
    rows = []
    for result in results_list:

        fastest_lap_time_str = None
        fastest_lap_seconds  = None
        fastest_lap_rank     = None

        if 'FastestLap' in result:
            fl = result['FastestLap']
            fastest_lap_time_str = fl.get('Time', {}).get('time')
            fastest_lap_seconds  = lap_time_to_seconds(fastest_lap_time_str)
            fastest_lap_rank     = int(fl.get('rank', 0))

        race_time_str     = result.get('Time', {}).get('time')
        race_time_seconds = lap_time_to_seconds(race_time_str)

        position_text   = result.get('positionText', '')
        finish_position = int(result['position']) if position_text.isdigit() else 0

        status, dnf_reason = parse_status(result.get('status', ''), position_text)

        rows.append({
            'season':              SEASON,
            'race_round':          int(race_info['round']),
            'race_name':           race_info['raceName'],
            'track_id':            TRACK_ID,
            'race_date':           race_info['date'],
            'session':             session,
            'driver_id':           result['Driver']['driverId'],
            'driver_name':         f"{result['Driver']['givenName']} {result['Driver']['familyName']}",
            'constructor_id':      result['Constructor']['constructorId'],
            'constructor_name':    result['Constructor']['name'],
            'grid_position':       int(result.get('grid', 0)),
            'finish_position':     finish_position,
            'points':              float(result.get('points', 0)),
            'laps_completed':      int(result.get('laps', 0)),
            'race_time_seconds':   race_time_seconds,
            'race_time_display':   race_time_str,
            'fastest_lap_seconds': fastest_lap_seconds,
            'fastest_lap_display': fastest_lap_time_str,
            'fastest_lap_rank':    fastest_lap_rank,
            'status':              status,
            'dnf_reason':          dnf_reason,
        })

    return pd.DataFrame(rows)


# ── STEP 1: FETCH MAIN RACE RESULTS ──────────────────────────────────────────
print(f"── Fetching Race Results: {SEASON} Round {ROUND} ──")

race_data = fetch_api(f"{BASE_URL}/{SEASON}/{ROUND}/results.json?limit=30")
df_race_results = pd.DataFrame()

if race_data:
    race_info    = race_data['MRData']['RaceTable']['Races'][0]
    results_list = race_info['Results']
    print(f"  ✓ Found: {race_info['raceName']} — {len(results_list)} drivers")
    df_race_results = build_results_df(results_list, race_info, session='Race')
else:
    print("  ✗ Could not fetch race results")

time.sleep(1)


# ── STEP 2: FETCH SPRINT RESULTS (sprint weekends only) ──────────────────────
df_sprint_results = pd.DataFrame()

if IS_SPRINT_WEEKEND:
    print()
    print(f"── Fetching Sprint Results: {SEASON} Round {ROUND} ──")

    sprint_data = fetch_api(f"{BASE_URL}/{SEASON}/{ROUND}/sprint.json?limit=30")

    if sprint_data:
        sprint_info  = sprint_data['MRData']['RaceTable']['Races'][0]
        sprint_list  = sprint_info['SprintResults']
        print(f"  ✓ Found: {len(sprint_list)} sprint results")

        df_sprint_results = build_results_df(sprint_list, sprint_info, session='Sprint')

        print()
        print("  Sprint top 5:")
        print(df_sprint_results[['driver_name', 'finish_position', 'points']]
              .head(5).to_string(index=False))
    else:
        print("  ✗ Could not fetch sprint results")

    time.sleep(1)


# ── STEP 3: COMBINE RACE + SPRINT AND SAVE ────────────────────────────────────
# Both sessions go into one CSV per weekend.
# The notebook's groupby().sum() on points will correctly add
# race points + sprint points into the cumulative standings.
print()
print(f"── Saving Race CSV ──")

frames      = [df for df in [df_race_results, df_sprint_results] if not df.empty]
df_combined = pd.concat(frames, ignore_index=True)

race_filename = f"r{ROUND:02d}_{TRACK_ID}_{SEASON}.csv"
df_combined.to_csv(os.path.join(RAW_DATA_PATH, race_filename), index=False)

print(f"  ✓ Saved: {race_filename}")
print(f"    Race rows:   {len(df_race_results)}")
if IS_SPRINT_WEEKEND:
    print(f"    Sprint rows: {len(df_sprint_results)}")
print(f"    Total rows:  {len(df_combined)}")


# ── STEP 4: FETCH QUALIFYING ──────────────────────────────────────────────────
print()
print(f"── Fetching Qualifying Results: {SEASON} Round {ROUND} ──")

qual_data = fetch_api(f"{BASE_URL}/{SEASON}/{ROUND}/qualifying.json?limit=30")

if qual_data:
    qual_info = qual_data['MRData']['RaceTable']['Races'][0]
    qual_list = qual_info['QualifyingResults']
    print(f"  ✓ Found: {len(qual_list)} qualifying results")

    rows = []
    for result in qual_list:
        q1_str = result.get('Q1')
        q2_str = result.get('Q2')
        q3_str = result.get('Q3')

        rows.append({
            'season':           SEASON,
            'race_round':       int(qual_info['round']),
            'race_name':        qual_info['raceName'],
            'track_id':         TRACK_ID,
            'race_date':        qual_info['date'],
            'driver_id':        result['Driver']['driverId'],
            'driver_name':      f"{result['Driver']['givenName']} {result['Driver']['familyName']}",
            'constructor_id':   result['Constructor']['constructorId'],
            'constructor_name': result['Constructor']['name'],
            'q1_seconds':       lap_time_to_seconds(q1_str),
            'q2_seconds':       lap_time_to_seconds(q2_str),
            'q3_seconds':       lap_time_to_seconds(q3_str),
            'pole_lap_seconds': None,
            'pole_lap_display': None,
            'grid_position':    int(result['position']),
        })

    df_qualifying = pd.DataFrame(rows)

    # Fill pole lap from P1 driver's best time
    pole_display = next(
        r.get('Q3') or r.get('Q2') or r.get('Q1')
        for r in qual_list if int(r['position']) == 1
    )
    pole_seconds = lap_time_to_seconds(pole_display)

    df_qualifying['pole_lap_seconds'] = pole_seconds
    df_qualifying['pole_lap_display'] = pole_display

    qual_filename = f"r{ROUND:02d}_{TRACK_ID}_{SEASON}_qualifying.csv"
    df_qualifying.to_csv(os.path.join(RAW_DATA_PATH, qual_filename), index=False)
    print(f"  ✓ Saved: {qual_filename} ({len(df_qualifying)} rows)")

else:
    print("  ✗ Could not fetch qualifying results")


# ── SUMMARY ───────────────────────────────────────────────────────────────────
print()
print("── Ingestion Complete ──")
print(f"  Race CSV:        {race_filename}")
if IS_SPRINT_WEEKEND:
    print(f"  Sprint included: Yes ({len(df_sprint_results)} rows)")
print(f"  Qualifying CSV:  {qual_filename}")
print()
print("Next step: Run Section 1 (Setup) through the rest of the notebook.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
── Fetching Race Results: 2026 Round 3 ──
  Fetching: http://api.jolpi.ca/ergast/f1/2026/3/results.json?limit=30
  ✓ Found: Japanese Grand Prix — 22 drivers

── Saving Race CSV ──
  ✓ Saved: r03_japan_2026.csv
    Race rows:   22
    Total rows:  22

── Fetching Qualifying Results: 2026 Round 3 ──
  Fetching: http://api.jolpi.ca/ergast/f1/2026/3/qualifying.json?limit=30
  ✓ Found: 22 qualifying results
  ✓ Saved: r03_japan_2026_qualifying.csv (22 rows)

── Ingestion Complete ──
  Race CSV:        r03_japan_2026.csv
  Qualifying CSV:  r03_japan_2026_qualifying.csv

Next step: Run Section 1 (Setup) through the rest of the notebook.


# 🏎️ F1 2026 Season Tracker

**Project:** Formula 1 2026 Season Analysis  
**Author:** Clay  
**Last Updated:** Update this each race weekend

---

## Notebook Structure
1. **Setup** — imports and configuration
2. **Data Ingestion** — load race CSVs and build season master
3. **Driver Standings** — cumulative points table
4. **Constructor Standings** — team points table
5. **Track Comparison** — 2026 vs 2022 lap time analysis
6. **Visualizations** — Plotly charts
7. **Tableau Export** — generate extract CSV

---
### 📋 Race Log — Update each week
| Round | Race | Date | Status |
|---|---|---|---|
| R01 | Australian GP | 2026-03-07 | ✅ Complete |
| R02 | Chinese Sprint | 2026-03-14 | ✅ Complete |
| R02 | Chinese GP | 2026-03-15 | ✅ Complete |
| R03 | Japanese GP | 2026-03-28 | ✅ Complete |
| R04 | Bahrain GP | 2026-04-12 | ❌ Canceled |
| R05 | Saudi Arabian GP | 2026-04-19 | ❌ Canceled |
| R06 | Miami GP | 2026-05-03 | ⏰ Up Next |

---
## Section 1: Setup
Run this cell first every time you open the notebook. It imports all libraries and sets global configuration.

In [ ]:
# ── IMPORTS ──────────────────────────────────────────────────────────────────
import pandas as pd          # Core data manipulation
import numpy as np           # Numerical operations
import plotly.express as px  # High-level Plotly charts
import plotly.graph_objects as go  # Low-level Plotly for custom charts
import os                    # File path operations
import glob                  # Find multiple files matching a pattern
from datetime import datetime

# ── DISPLAY SETTINGS ─────────────────────────────────────────────────────────
# Show all columns when printing DataFrames — important for wide tables
pd.set_option('display.max_columns', None)
# Show full content of each cell, not truncated with '...'
pd.set_option('display.max_colwidth', None)
# Limit rows shown to keep output manageable
pd.set_option('display.max_rows', 50)

# ── FILE PATHS ────────────────────────────────────────────────────────────────
# Using relative paths so this works on any machine after cloning from GitHub.
# Everything is relative to the /notebooks/ folder where this file lives.
RAW_DATA_PATH  = '/content/drive/MyDrive/f1-2026-season/data/raw/'           # Per-race CSVs go here
PROCESSED_PATH = '/content/drive/MyDrive/f1-2026-season/data/processed/'     # Merged season master lives here
BASELINE_PATH  = '/content/drive/MyDrive/f1-2026-season/data/baseline/'      # 2022 comparison data
EXPORTS_PATH   = '/content/drive/MyDrive/f1-2026-season/exports/'            # Tableau extract goes here

print(f"Setup complete — {datetime.now().strftime('%Y-%m-%d %H:%M')}")

Setup complete — 2026-03-31 04:33


---
## Section 2: Data Ingestion

This section does the heavy lifting of reading every race CSV and merging them into two master DataFrames — one for race results, one for qualifying.

**Key concept:** `glob.glob()` finds all files matching a pattern (like `r*_2026.csv`), so you never have to manually list filenames. As you add new race CSVs, this code picks them up automatically.

In [ ]:
# ── LOAD ALL RACE RESULT FILES ────────────────────────────────────────────────
# glob finds every file matching the pattern — the * is a wildcard
# sorted() ensures they're processed in round order (r01, r02, r03...)
race_files = sorted(glob.glob(RAW_DATA_PATH + 'r*_2026.csv'))

# Tell us what was found — useful for debugging
print(f"Found {len(race_files)} race result file(s):")
for f in race_files:
    print(f"  {os.path.basename(f)}")

# Read each CSV into its own DataFrame, collect them in a list
# This is the pattern you'll use whenever you need to process multiple files
race_dfs = []
for file in race_files:
    df = pd.read_csv(file)
    race_dfs.append(df)

# pd.concat() stacks DataFrames vertically (on top of each other)
# ignore_index=True resets the row index so it runs 0, 1, 2... continuously
# instead of restarting at 0 for each race
df_races = pd.concat(race_dfs, ignore_index=True)

print(f"\nSeason master shape: {df_races.shape[0]} rows x {df_races.shape[1]} columns")
print(f"Rounds loaded: {sorted(df_races['race_round'].unique())}")

Found 3 race result file(s):
  r01_australia_2026.csv
  r02_china_2026.csv
  r03_japan_2026.csv

Season master shape: 88 rows x 21 columns
Rounds loaded: [np.int64(1), np.int64(2), np.int64(3)]


In [ ]:
# Run this after df_races is built in Section 2
print(df_races[['driver_name', 'constructor_id', 'constructor_name']]
      .drop_duplicates()
      .sort_values('constructor_name'))

              driver_name constructor_id  constructor_name
13       Franco Colapinto         alpine    Alpine F1 Team
9            Pierre Gasly         alpine    Alpine F1 Team
17        Fernando Alonso   aston_martin      Aston Martin
16           Lance Stroll   aston_martin      Aston Martin
21        Nico Hülkenberg           audi              Audi
8       Gabriel Bortoleto           audi              Audi
18        Valtteri Bottas       cadillac  Cadillac F1 Team
15           Sergio Pérez       cadillac  Cadillac F1 Team
2         Charles Leclerc        ferrari           Ferrari
3          Lewis Hamilton        ferrari           Ferrari
10           Esteban Ocon           haas      Haas F1 Team
6          Oliver Bearman           haas      Haas F1 Team
4            Lando Norris        mclaren           McLaren
20          Oscar Piastri        mclaren           McLaren
1   Andrea Kimi Antonelli       mercedes          Mercedes
0          George Russell       mercedes          Merced

In [ ]:
# ── LOAD ALL QUALIFYING FILES ─────────────────────────────────────────────────
qual_files = sorted(glob.glob(RAW_DATA_PATH + 'r*_qualifying.csv'))

print(f"Found {len(qual_files)} qualifying file(s):")
for f in qual_files:
    print(f"  {os.path.basename(f)}")

qual_dfs = []
for file in qual_files:
    df = pd.read_csv(file)
    qual_dfs.append(df)

df_qualifying = pd.concat(qual_dfs, ignore_index=True)

print(f"\nQualifying master shape: {df_qualifying.shape[0]} rows x {df_qualifying.shape[1]} columns")

Found 3 qualifying file(s):
  r01_australia_2026_qualifying.csv
  r02_china_2026_qualifying.csv
  r03_japan_2026_qualifying.csv

Qualifying master shape: 63 rows x 15 columns


In [ ]:
# ── QUICK DATA HEALTH CHECK ───────────────────────────────────────────────────
# Always inspect your data after loading — catch problems early

print("=== RACE DATA SAMPLE ===")
print(df_races.head(5))

print("\n=== DATA TYPES ===")
# df.dtypes tells you how pandas interpreted each column
# If a number column shows 'object', that means pandas read it as text — a common problem
print(df_races.dtypes)

print("\n=== NULL VALUE COUNT ===")
# Nulls are expected in some columns (dnf_reason, race_time for DNFs)
# but unexpected nulls in key columns like points or finish_position are a problem
print(df_races.isnull().sum())

=== RACE DATA SAMPLE ===
   season  race_round              race_name   track_id   race_date  \
0    2026           1  Australian Grand Prix  australia  2026-03-08   
1    2026           1  Australian Grand Prix  australia  2026-03-08   
2    2026           1  Australian Grand Prix  australia  2026-03-08   
3    2026           1  Australian Grand Prix  australia  2026-03-08   
4    2026           1  Australian Grand Prix  australia  2026-03-08   

   driver_id            driver_name constructor_id constructor_name  \
0    russell         George Russell       mercedes         Mercedes   
1  antonelli  Andrea Kimi Antonelli       mercedes         Mercedes   
2    leclerc        Charles Leclerc        ferrari          Ferrari   
3   hamilton         Lewis Hamilton        ferrari          Ferrari   
4     norris           Lando Norris        mclaren          McLaren   

   grid_position  finish_position  points  laps_completed  race_time_seconds  \
0              1                1    25.0

In [ ]:
# ── SAVE SEASON MASTER TO PROCESSED FOLDER ────────────────────────────────────
# This gives you a single merged file you can reload quickly
# without re-running the ingestion loop every time
df_races.to_csv(PROCESSED_PATH + 'season_master_2026.csv', index=False)
df_qualifying.to_csv(PROCESSED_PATH + 'qualifying_master_2026.csv', index=False)

print("Season master files saved to /data/processed/")

Season master files saved to /data/processed/


---
## Section 3: Driver Standings

Here we derive the championship table from raw race results using `groupby()` and aggregation.

**Key concept:** We never stored 'current points total' in our CSVs — we calculate it here. This means the standings are always accurate to whatever races are loaded, with no risk of manual entry errors.

In [ ]:
# ── DRIVER STANDINGS TABLE ────────────────────────────────────────────────────

# groupby() groups all rows by driver, then .agg() defines what to do with each column
# Think of this like a SQL GROUP BY with multiple aggregation functions
df_driver_standings = (
    df_races
    .groupby(['driver_id', 'driver_name', 'constructor_name'])
    .agg(
        total_points   = ('points', 'sum'),        # Sum all points scored
        races_entered  = ('race_round', 'count'),  # Count rows = races entered
        wins           = ('finish_position',        # Count rows where position == 1
                          lambda x: (x == 1).sum()),
        podiums        = ('finish_position',
                          lambda x: (x <= 3).sum()),
        dnfs           = ('status',
                          lambda x: (x == 'DNF').sum()),
        best_finish    = ('finish_position',
                          lambda x: x[x > 0].min()),  # Exclude DNFs (position = 0)
        fastest_laps   = ('fastest_lap_rank',
                          lambda x: (x == 1).sum()),
    )
    .reset_index()  # Moves groupby columns back into regular columns
    .sort_values('total_points', ascending=False)  # Highest points first
    .reset_index(drop=True)  # Clean 0-based index after sorting
)

# Add championship position as its own column
# The index is 0-based so we add 1 to get positions 1, 2, 3...
df_driver_standings.insert(0, 'position', df_driver_standings.index + 1)

print("=== 2026 DRIVER CHAMPIONSHIP STANDINGS ===")
print(df_driver_standings.to_string(index=False))

=== 2026 DRIVER CHAMPIONSHIP STANDINGS ===
 position      driver_id           driver_name constructor_name  total_points  races_entered  wins  podiums  dnfs  best_finish  fastest_laps
        1      antonelli Andrea Kimi Antonelli         Mercedes          72.0              4     2        3     0            1             2
        2        russell        George Russell         Mercedes          63.0              4     2        3     0            1             0
        3        leclerc       Charles Leclerc          Ferrari          49.0              4     0        3     0            2             1
        4       hamilton        Lewis Hamilton          Ferrari          41.0              4     0        2     0            3             0
        5         norris          Lando Norris          McLaren          25.0              4     0        1     1            4             0
        6        piastri         Oscar Piastri          McLaren          21.0              4     0        3    

In [ ]:
# ── CUMULATIVE POINTS BY ROUND ────────────────────────────────────────────────
# This builds the data needed for a championship progression chart
# showing how each driver's points total grows round by round

# Sort so cumsum() runs in the correct chronological order per driver
df_cumulative = df_races.sort_values(['driver_id', 'race_round']).copy()

# cumsum() within each driver group — running total of points
# groupby().cumsum() is the pandas equivalent of a SQL window function:
# SUM(points) OVER (PARTITION BY driver_id ORDER BY race_round)
df_cumulative['cumulative_points'] = (
    df_cumulative.groupby('driver_id')['points'].cumsum()
)

print("=== CUMULATIVE POINTS SAMPLE (first 10 rows) ===")
print(df_cumulative[['race_round', 'race_name', 'driver_name', 'points', 'cumulative_points']].head(10))

=== CUMULATIVE POINTS SAMPLE (first 10 rows) ===
    race_round              race_name            driver_name  points  \
11           1  Australian Grand Prix        Alexander Albon     0.0   
43           2     Chinese Grand Prix        Alexander Albon     0.0   
59           2     Chinese Grand Prix        Alexander Albon     0.0   
85           3    Japanese Grand Prix        Alexander Albon     0.0   
17           1  Australian Grand Prix        Fernando Alonso     0.0   
38           2     Chinese Grand Prix        Fernando Alonso     0.0   
60           2     Chinese Grand Prix        Fernando Alonso     0.0   
83           3    Japanese Grand Prix        Fernando Alonso     0.0   
1            1  Australian Grand Prix  Andrea Kimi Antonelli    18.0   
22           2     Chinese Grand Prix  Andrea Kimi Antonelli    25.0   

    cumulative_points  
11                0.0  
43                0.0  
59                0.0  
85                0.0  
17                0.0  
38            

---
## Section 4: Constructor Standings

Constructors score points from both their drivers combined. The logic is similar to driver standings but we group by team instead.

In [ ]:
# ── CONSTRUCTOR STANDINGS TABLE ───────────────────────────────────────────────

df_constructor_standings = (
    df_races
    .groupby(['constructor_id', 'constructor_name'])
    .agg(
        total_points  = ('points', 'sum'),
        wins          = ('finish_position', lambda x: (x == 1).sum()),
        podiums       = ('finish_position', lambda x: (x <= 3).sum()),
        dnfs          = ('status', lambda x: (x == 'DNF').sum()),
        fastest_laps  = ('fastest_lap_rank', lambda x: (x == 1).sum()),
    )
    .reset_index()
    .sort_values('total_points', ascending=False)
    .reset_index(drop=True)
)

df_constructor_standings.insert(0, 'position', df_constructor_standings.index + 1)

print("=== 2026 CONSTRUCTOR CHAMPIONSHIP STANDINGS ===")
print(df_constructor_standings.to_string(index=False))

=== 2026 CONSTRUCTOR CHAMPIONSHIP STANDINGS ===
 position constructor_id constructor_name  total_points  wins  podiums  dnfs  fastest_laps
        1       mercedes         Mercedes         135.0     4        6     0             2
        2        ferrari          Ferrari          90.0     0        5     0             1
        3        mclaren          McLaren          46.0     0        4     3             0
        4           haas     Haas F1 Team          18.0     0        1     1             0
        5       red_bull         Red Bull          16.0     0        2     2             1
        6         alpine   Alpine F1 Team          16.0     0        0     0             0
        7             rb       RB F1 Team          14.0     0        1     1             0
        8       williams         Williams           2.0     0        1     1             0
        9           audi             Audi           2.0     0        3     3             0
       10   aston_martin     Aston Martin 

In [ ]:
# ── POINTS BREAKDOWN BY DRIVER PER CONSTRUCTOR ────────────────────────────────
# Shows how much each driver contributed to their team's total
# Useful for understanding if one driver is carrying the team

df_constructor_breakdown = (
    df_races
    .groupby(['constructor_name', 'driver_name'])
    .agg(total_points=('points', 'sum'))
    .reset_index()
    .sort_values(['constructor_name', 'total_points'], ascending=[True, False])
)

print("=== POINTS BY DRIVER PER TEAM ===")
print(df_constructor_breakdown.to_string(index=False))

=== POINTS BY DRIVER PER TEAM ===
constructor_name           driver_name  total_points
  Alpine F1 Team          Pierre Gasly          15.0
  Alpine F1 Team      Franco Colapinto           1.0
    Aston Martin       Fernando Alonso           0.0
    Aston Martin          Lance Stroll           0.0
            Audi     Gabriel Bortoleto           2.0
            Audi       Nico Hülkenberg           0.0
Cadillac F1 Team          Sergio Pérez           0.0
Cadillac F1 Team       Valtteri Bottas           0.0
         Ferrari       Charles Leclerc          49.0
         Ferrari        Lewis Hamilton          41.0
    Haas F1 Team        Oliver Bearman          17.0
    Haas F1 Team          Esteban Ocon           1.0
         McLaren          Lando Norris          25.0
         McLaren         Oscar Piastri          21.0
        Mercedes Andrea Kimi Antonelli          72.0
        Mercedes        George Russell          63.0
      RB F1 Team           Liam Lawson          10.0
      RB F1 

---
## Section 5: Track Comparison — 2026 vs 2022

This is the most analytically interesting part of the project. We join the 2026 race data to the 2022 baseline on `track_id` to compare lap times across the two regulatory eras.

**Key concept:** We use `how='left'` on the merge so that 2026 races without a 2022 equivalent (new circuits) still appear in the table — they'll just have nulls in the 2022 columns.

In [ ]:
# ── LOAD 2022 BASELINE ────────────────────────────────────────────────────────
df_2022 = pd.read_csv(BASELINE_PATH + '2022_season.csv')

print(f"2022 baseline loaded: {df_2022.shape[0]} races")
print(df_2022[['track_id', 'race_name', 'pole_lap_display', 'fastest_race_lap_display']].to_string(index=False))

2022 baseline loaded: 20 races
   track_id                 race_name pole_lap_display fastest_race_lap_display
  australia     Australian Grand Prix         1:19.813                 1:20.260
      imola Emilia Romagna Grand Prix         1:15.484                 1:17.069
      miami          Miami Grand Prix         1:31.361                 1:33.562
      spain        Spanish Grand Prix         1:19.526                 1:21.670
     monaco         Monaco Grand Prix         1:11.376                 1:17.774
 azerbaijan     Azerbaijan Grand Prix         1:22.046                 1:23.357
     canada       Canadian Grand Prix         1:13.044                 1:15.749
    britain        British Grand Prix         1:25.466                 1:27.097
    austria       Austrian Grand Prix         1:03.833                 1:06.200
     france         French Grand Prix         1:29.346                 1:31.346
    hungary      Hungarian Grand Prix         1:16.072                 1:18.446
    belgi

In [ ]:
# ── BUILD 2026 TRACK SUMMARY ──────────────────────────────────────────────────
# Pull the key per-race metrics we need from 2026 data
# We need pole time (from qualifying) and fastest race lap + winner time (from races)

# From qualifying: get pole position lap time per race
df_pole_2026 = (
    df_qualifying[df_qualifying['grid_position'] == 1]
    [['track_id', 'race_round', 'race_name', 'pole_lap_seconds', 'pole_lap_display',
      'driver_name']]
    .rename(columns={'driver_name': 'pole_sitter_2026'})
)

# From races: get fastest lap and winner's total race time
df_race_metrics_2026 = (
    df_races[df_races['fastest_lap_rank'] == 1]
    [['track_id', 'fastest_lap_seconds', 'fastest_lap_display', 'driver_name']]
    .rename(columns={
        'fastest_lap_seconds': 'fastest_race_lap_seconds_2026',
        'fastest_lap_display': 'fastest_race_lap_display_2026',
        'driver_name': 'fastest_lap_driver_2026'
    })
)

# Get winner's total race time
df_winner_2026 = (
    df_races[df_races['finish_position'] == 1]
    [['track_id', 'race_time_seconds', 'race_time_display', 'driver_name']]
    .rename(columns={
        'race_time_seconds': 'winner_race_time_seconds_2026',
        'race_time_display': 'winner_race_time_display_2026',
        'driver_name': 'winner_2026'
    })
)

# Merge the 2026 components together on track_id
df_2026_summary = (
    df_pole_2026
    .merge(df_race_metrics_2026, on='track_id', how='left')
    .merge(df_winner_2026, on='track_id', how='left')
)

print("=== 2026 TRACK SUMMARY ===")
print(df_2026_summary.to_string(index=False))

=== 2026 TRACK SUMMARY ===
 track_id  race_round             race_name  pole_lap_seconds pole_lap_display      pole_sitter_2026  fastest_race_lap_seconds_2026 fastest_race_lap_display_2026 fastest_lap_driver_2026  winner_race_time_seconds_2026 winner_race_time_display_2026           winner_2026
australia           1 Australian Grand Prix            78.518         1:18.518        George Russell                         82.091                      1:22.091          Max Verstappen                       4986.801                   1:23:06.801        George Russell
    china           2    Chinese Grand Prix            92.064         1:32.064 Andrea Kimi Antonelli                         95.275                      1:35.275   Andrea Kimi Antonelli                       5595.607                   1:33:15.607 Andrea Kimi Antonelli
    china           2    Chinese Grand Prix            92.064         1:32.064 Andrea Kimi Antonelli                         95.275                      1:35.275   An

In [ ]:
# ── MERGE 2026 vs 2022 ────────────────────────────────────────────────────────
# Rename 2022 columns to make the join clean and avoid name collisions
df_2022_slim = df_2022.rename(columns={
    'pole_lap_seconds':            'pole_lap_seconds_2022',
    'pole_lap_display':            'pole_lap_display_2022',
    'fastest_race_lap_seconds':    'fastest_race_lap_seconds_2022',
    'fastest_race_lap_display':    'fastest_race_lap_display_2022',
    'winner_race_time_seconds':    'winner_race_time_seconds_2022',
    'winner_race_time_display':    'winner_race_time_display_2022',
    'winner_driver_name':          'winner_2022',
})[['track_id', 'pole_lap_seconds_2022', 'pole_lap_display_2022',
    'fastest_race_lap_seconds_2022', 'fastest_race_lap_display_2022',
    'winner_race_time_seconds_2022', 'winner_race_time_display_2022',
    'winner_2022']]

# Left join: keep all 2026 races, match 2022 where track_id aligns
# Tracks new to 2026 that weren't on the 2022 calendar will have NaN in 2022 columns
df_track_comparison = df_2026_summary.merge(df_2022_slim, on='track_id', how='left')

# ── CALCULATE DELTAS ──────────────────────────────────────────────────────────
# Negative delta = 2026 is FASTER (smaller number = quicker lap)
# Positive delta = 2026 is SLOWER
df_track_comparison['pole_delta_seconds'] = (
    df_track_comparison['pole_lap_seconds'] - df_track_comparison['pole_lap_seconds_2022']
).round(3)

df_track_comparison['fastest_lap_delta_seconds'] = (
    df_track_comparison['fastest_race_lap_seconds_2026'] - df_track_comparison['fastest_race_lap_seconds_2022']
).round(3)

df_track_comparison['race_time_delta_seconds'] = (
    df_track_comparison['winner_race_time_seconds_2026'] - df_track_comparison['winner_race_time_seconds_2022']
).round(3)

# Display the comparison table
comparison_display_cols = [
    'race_name', 'track_id',
    'pole_lap_display_2022', 'pole_lap_display',
    'pole_delta_seconds',
    'fastest_race_lap_display_2022', 'fastest_race_lap_display_2026',
    'fastest_lap_delta_seconds',
    'winner_race_time_display_2022', 'winner_race_time_display_2026',
    'race_time_delta_seconds'
]

print("=== 2026 vs 2022 TRACK COMPARISON ===")
print("Negative delta = 2026 is faster | Positive delta = 2026 is slower")
print()
print(df_track_comparison[comparison_display_cols].to_string(index=False))

=== 2026 vs 2022 TRACK COMPARISON ===
Negative delta = 2026 is faster | Positive delta = 2026 is slower

            race_name  track_id pole_lap_display_2022 pole_lap_display  pole_delta_seconds fastest_race_lap_display_2022 fastest_race_lap_display_2026  fastest_lap_delta_seconds winner_race_time_display_2022 winner_race_time_display_2026  race_time_delta_seconds
Australian Grand Prix australia              1:19.813         1:18.518              -1.295                      1:20.260                      1:22.091                      1.831                   1:30:13.121                   1:23:06.801                 -426.320
   Chinese Grand Prix     china                   NaN         1:32.064                 NaN                           NaN                      1:35.275                        NaN                           NaN                   1:33:15.607                      NaN
   Chinese Grand Prix     china                   NaN         1:32.064                 NaN                

---
## Section 6: Visualizations

All charts use Plotly Express — interactive, hoverable, and exportable. Phase 2 will expand these significantly once we have more rounds of data.

In [ ]:
# ── CHART 1: DRIVER STANDINGS BAR CHART ──────────────────────────────────────

fig_driver_standings = px.bar(
    df_driver_standings,
    x='driver_name',
    y='total_points',
    color='constructor_name',
    title='2026 F1 Driver Championship Standings',
    labels={
        'driver_name': 'Driver',
        'total_points': 'Total Points',
        'constructor_name': 'Constructor'
    },
    text='total_points',           # Show point value on each bar
    category_orders={'driver_name': df_driver_standings['driver_name'].tolist()}
)

fig_driver_standings.update_layout(
    xaxis_tickangle=-45,
    showlegend=True,
    height=500
)

fig_driver_standings.show()

In [ ]:
# ── CHART 2: CONSTRUCTOR STANDINGS BAR CHART ──────────────────────────────────

fig_constructor_standings = px.bar(
    df_constructor_standings,
    x='constructor_name',
    y='total_points',
    color='constructor_name',
    title='2026 F1 Constructor Championship Standings',
    labels={
        'constructor_name': 'Constructor',
        'total_points': 'Total Points'
    },
    text='total_points',
    category_orders={'constructor_name': df_constructor_standings['constructor_name'].tolist()}
)

fig_constructor_standings.update_layout(
    xaxis_tickangle=-30,
    showlegend=False,
    height=500
)

fig_constructor_standings.show()

In [ ]:
# ── CHART 3: CHAMPIONSHIP PROGRESSION (needs 2+ rounds) ───────────────────────
# This chart will populate fully once you have Race 2 data loaded
# With only one round it shows a single point per driver

# Get top 10 drivers by total points for a cleaner chart
top_10_drivers = df_driver_standings.head(10)['driver_name'].tolist()
df_cumulative_top10 = df_cumulative[df_cumulative['driver_name'].isin(top_10_drivers)]

fig_progression = px.line(
    df_cumulative_top10,
    x='race_round',
    y='cumulative_points',
    color='driver_name',
    markers=True,              # Show dots at each data point
    title='2026 F1 Championship Progression — Top 10 Drivers',
    labels={
        'race_round': 'Race Round',
        'cumulative_points': 'Cumulative Points',
        'driver_name': 'Driver'
    },
    hover_data=['race_name']   # Show race name when hovering
)

fig_progression.update_layout(height=500)
fig_progression.show()

In [ ]:
# ── CHART 4: TRACK COMPARISON — POLE LAP DELTA ────────────────────────────────
# Only shows tracks where we have both 2022 and 2026 data

df_comparison_clean = df_track_comparison.dropna(subset=['pole_delta_seconds'])

# Color bars: green if 2026 faster, red if slower
df_comparison_clean = df_comparison_clean.copy()
df_comparison_clean['faster_in_2026'] = df_comparison_clean['pole_delta_seconds'] < 0

fig_pole_delta = px.bar(
    df_comparison_clean,
    x='track_id',
    y='pole_delta_seconds',
    color='faster_in_2026',
    color_discrete_map={True: '#00C851', False: '#ff4444'},
    title='Pole Lap Delta: 2026 vs 2022 (Negative = 2026 Faster)',
    labels={
        'track_id': 'Circuit',
        'pole_delta_seconds': 'Delta (seconds)',
        'faster_in_2026': '2026 Faster'
    },
    hover_data=['pole_lap_display_2022', 'pole_lap_display']
)

fig_pole_delta.add_hline(y=0, line_dash='dash', line_color='white', opacity=0.5)
fig_pole_delta.update_layout(height=450)
fig_pole_delta.show()

---
## Section 7: Tableau Export

We export a single flattened CSV that contains everything Tableau needs to build all three views — standings, track comparison, and progression. One file keeps the Tableau workbook simple.

In [ ]:
# ── BUILD TABLEAU EXTRACT ─────────────────────────────────────────────────────
# Strategy: export three separate sheets as separate CSVs
# Tableau can blend multiple data sources, or you can union them in Tableau Prep

# 1. Full race results with cumulative points — the main fact table
tableau_race_data = df_cumulative[[
    'season', 'race_round', 'race_name', 'track_id', 'race_date',
    'driver_id', 'driver_name', 'constructor_id', 'constructor_name',
    'grid_position', 'finish_position', 'points', 'cumulative_points',
    'fastest_lap_seconds', 'fastest_lap_rank', 'status', 'dnf_reason'
]]

# 2. Driver standings snapshot
tableau_driver_standings = df_driver_standings.copy()

# 3. Constructor standings snapshot
tableau_constructor_standings = df_constructor_standings.copy()

# 4. Track comparison
tableau_track_comparison = df_track_comparison[[
    'track_id', 'race_name', 'race_round',
    'pole_lap_seconds', 'pole_lap_display',
    'pole_lap_seconds_2022', 'pole_lap_display_2022',
    'pole_delta_seconds',
    'fastest_race_lap_seconds_2026', 'fastest_race_lap_display_2026',
    'fastest_race_lap_seconds_2022', 'fastest_race_lap_display_2022',
    'fastest_lap_delta_seconds',
    'winner_race_time_seconds_2026', 'winner_race_time_display_2026',
    'winner_race_time_seconds_2022', 'winner_race_time_display_2022',
    'race_time_delta_seconds'
]]

# Export all four
tableau_race_data.to_csv(EXPORTS_PATH + 'tableau_race_results_2026.csv', index=False)
tableau_driver_standings.to_csv(EXPORTS_PATH + 'tableau_driver_standings_2026.csv', index=False)
tableau_constructor_standings.to_csv(EXPORTS_PATH + 'tableau_constructor_standings_2026.csv', index=False)
tableau_track_comparison.to_csv(EXPORTS_PATH + 'tableau_track_comparison_2026.csv', index=False)

timestamp = datetime.now().strftime('%Y-%m-%d %H:%M')
print(f"Tableau exports complete — {timestamp}")
print(f"  tableau_race_results_2026.csv       — {len(tableau_race_data)} rows")
print(f"  tableau_driver_standings_2026.csv   — {len(tableau_driver_standings)} rows")
print(f"  tableau_constructor_standings_2026.csv — {len(tableau_constructor_standings)} rows")
print(f"  tableau_track_comparison_2026.csv   — {len(tableau_track_comparison)} rows")

Tableau exports complete — 2026-03-31 04:33
  tableau_race_results_2026.csv       — 88 rows
  tableau_driver_standings_2026.csv   — 22 rows
  tableau_constructor_standings_2026.csv — 11 rows
  tableau_track_comparison_2026.csv   — 6 rows


---
## Section 8: Weekly Race Update Checklist

Copy and run these steps each race weekend.

```
□ 1. Create new race CSV:     data/raw/r0X_racename_2026.csv
□ 2. Create qualifying CSV:   data/raw/r0X_racename_2026_qualifying.csv
□ 3. Update Race Log table at top of notebook
□ 4. Run All Cells (Runtime → Run all)
□ 5. Review standings tables for accuracy
□ 6. Review track comparison if this circuit has a 2022 equivalent
□ 7. Check /exports/ folder for updated Tableau CSVs
□ 8. Upload Tableau CSVs to Tableau workbook
□ 9. Git commit and push:
       git add .
       git commit -m "Add Race X - [Race Name] results"
       git push
```

---
*Next phase: Replace manual CSV ingestion with Ergast/OpenF1 API calls.*